# Aesha's contribution

In [1]:
pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.9/395.9 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.0/247.0 kB 21.0 MB/s eta 0:00:00


In [2]:
pip install transformers datasets peft accelerate evaluate bert_score rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 70.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 27.7 MB/s eta 0:00:00
  Crea

In [ ]:
import optuna
import pandas as pd
from transformers import Trainer, TrainingArguments
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
import torch
from datasets import Dataset

# Load training passages and test QA data
passages_df = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/passages.parquet/part.0.parquet")
test_df = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/test.parquet/part.0.parquet")

# Sample data
train_data_sample = passages_df.sample(frac=0.1, random_state=42)
train_dataset = Dataset.from_pandas(train_data_sample[['passage']].rename(columns={'passage': 'text'}))
test_dataset = Dataset.from_pandas(test_df[['question']].rename(columns={'question': 'text'}))

# Load GPT-2 model and tokenizer
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Set the pad token to eos_token if not set already
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token  # Set pad token to eos_token

# Apply LoRA (Low-Rank Adaptation) to the model
lora_config = LoraConfig(
    r=8,  # Rank of the low-rank adaptation
    lora_alpha=16,  # Scaling factor
    target_modules=["attn.c_attn", "attn.c_proj"],  # LoRA applied to these layers for GPT-2
    lora_dropout=0.1,  # Dropout for LoRA
)

# Apply LoRA to the GPT-2 model
model_lora = get_peft_model(model, lora_config)

# Tokenization function
def tokenize_data(examples):
    tokenized_inputs = tokenizer(examples['text'], truncation=True, padding=True, max_length=256, return_attention_mask=True)
    tokenized_inputs["labels"] = tokenized_inputs["input_ids"].copy()
    return tokenized_inputs

# Tokenize the passages and test data
train_data = train_dataset.map(tokenize_data, batched=True)
test_data = test_dataset.map(tokenize_data, batched=True)

# Define the function for the Optuna objective
def objective(trial):
    # Hyperparameter search space
    learning_rate = trial.suggest_loguniform('learning_rate', 1e-6, 1e-3)
    batch_size = trial.suggest_int('batch_size', 1, 8)  # Use smaller batch size
    num_train_epochs = trial.suggest_int('num_train_epochs', 3, 10)

    # Training Arguments with gradient accumulation and mixed precision enabled
    training_args = TrainingArguments(
        output_dir='./results',
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=num_train_epochs,
        weight_decay=0.01,
        logging_dir='./logs',
        logging_steps=500,
        gradient_accumulation_steps=4,  # Gradient accumulation for larger effective batch size
        fp16=True,  # Mixed precision training
    )

    # Trainer
    trainer = Trainer(
        model=model_lora,
        args=training_args,
        train_dataset=train_data,
        eval_dataset=test_data,
    )

    # Train the model
    trainer.train()

    # Evaluate the model
    results = trainer.evaluate()
    return results['eval_loss']  # Minimize the evaluation loss

# Create the Optuna study and optimize the hyperparameters
study = optuna.create_study(direction='minimize')  # Minimize the evaluation loss
study.optimize(objective, n_trials=5)  # You can change `n_trials` to test more hyperparameter sets

# Get the best trial (the optimal hyperparameters)
best_trial = study.best_trial
print("Best trial:")
print("  Value: ", best_trial.value)
print("  Params: ")
for key, value in best_trial.params.items():
    print(f"    {key}: {value}")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: taran16501 (taran16501-loyalist-college) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss


In [ ]:
# Best hyperparameters
best_lr = best_trial.params['learning_rate']
best_batch_size = best_trial.params['batch_size']
best_epochs = best_trial.params['num_train_epochs']

# Final model training with the best hyperparameters
final_training_args = TrainingArguments(
    output_dir='./results',
    learning_rate=best_lr,
    per_device_train_batch_size=best_batch_size,
    per_device_eval_batch_size=best_batch_size,
    num_train_epochs=best_epochs,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=500,
)

final_trainer = Trainer(
    model=model_lora,
    args=final_training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
)

final_trainer.train()

# Evaluate the model on the test set
final_results = final_trainer.evaluate()
print("Final Evaluation Results:", final_results)

# LLM Used
1. Perform Model tuning on RAG approach on pretained LLM.
2. Provide step by step solution from scratch starting from importing data to the final evaluation results.
3. Facing error the following error. what's the solution to overcome this:Tried to use `fp16` but it is not supported on cpu
